# CQP NetKet/JAX ViT VMC

This notebook mirrors the old `CQP MPS/main.ipynb` structure, but uses the JAX/NetKet-style backend in this folder. The default run is intentionally small so it can be handed off and sanity-checked before scaling up on the JAX server.

In [ ]:
import json
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

try:
    from tqdm.auto import trange
except ImportError:
    def trange(*args, **kwargs):
        return range(*args)
from netket_cqp import J1J2Hamiltonian, SROptimizer, VMCSampler, ViTWaveFunction

print("JAX devices:", jax.devices())
print("jax_enable_x64:", jax.config.x64_enabled)

## Helper functions

In [ ]:
def flatten_samples(states):
    """Flatten (n_samples, n_chains, L) -> (M, L)."""
    return states.reshape(-1, states.shape[-1])


def count_real_dof(params):
    """Count real trainable degrees of freedom, with complex leaves counted twice."""
    total = 0
    for leaf in jax.tree_util.tree_leaves(params):
        arr = jnp.asarray(leaf)
        total += arr.size * (2 if jnp.issubdtype(arr.dtype, jnp.complexfloating) else 1)
    return int(total)

def exact_ground_state_energy_sz0(L, J1=1.0, J2=0.0):
    """Exact periodic J1-J2 ground-state energy in the total-Sz=0 sector."""
    from scipy.sparse import dok_array
    from scipy.sparse.linalg import eigsh

    if L % 2 != 0:
        raise ValueError("This helper assumes an even chain length and total Sz=0.")

    basis = [state for state in range(1 << L) if state.bit_count() == L // 2]
    index = {state: i for i, state in enumerate(basis)}
    H = dok_array((len(basis), len(basis)), dtype=np.float64)

    pairs = [(i, (i + 1) % L, J1) for i in range(L)]
    if J2 != 0.0:
        pairs.extend((i, (i + 2) % L, J2) for i in range(L))

    for row, state in enumerate(basis):
        diag = 0.0
        for i, j, J in pairs:
            bit_i = (state >> i) & 1
            bit_j = (state >> j) & 1
            s_i = 0.5 if bit_i else -0.5
            s_j = 0.5 if bit_j else -0.5
            diag += J * s_i * s_j

            if bit_i != bit_j:
                flipped = state ^ ((1 << i) | (1 << j))
                H[row, index[flipped]] += 0.5 * J

        H[row, row] += diag

    return float(eigsh(H.tocsr(), k=1, which="SA", return_eigenvectors=False)[0])


def spin_correlation(configs):
    """C(r) = <s_i s_{i+r}> averaged over samples and sites."""
    s = np.asarray(configs, dtype=np.float64)
    L = s.shape[1]
    return np.array([np.mean(s * np.roll(s, -r, axis=1)) for r in range(L)])


def structure_factor_from_C(C):
    """S(k) from the spin correlation C(r)."""
    C = np.asarray(C)
    return np.fft.fft(C).real


def dimer_correlation(configs):
    """D(r) = <B_i B_{i+r}> with B_i = s_i s_{i+1}."""
    s = np.asarray(configs, dtype=np.float64)
    B = s * np.roll(s, -1, axis=1)
    L = s.shape[1]
    return np.array([np.mean(B * np.roll(B, -r, axis=1)) for r in range(L)])


def tree_l2_norm(params):
    """Simple real-valued norm for a parameter pytree."""
    total = 0.0
    for leaf in jax.tree_util.tree_leaves(params):
        arr = jnp.asarray(leaf)
        total += float(jnp.sum(jnp.abs(arr) ** 2))
    return total ** 0.5


## Parameters

In [ ]:
# Spin chain / Hamiltonian
L = 16
J1 = 1.0
J2_ratio = 0.4

# ViT ansatz. Here d and h follow the requested convention.
b = 4
d = 16
h = 4
n_layers = 2
d_model = d
n_heads = h
d_ff = 64
rbm_hidden = 32
amp_init_std = 0.05
phase_init_std = 0.1
J2 = J2_ratio * J1
# Keep this off for the complex ansatz: the shift-sum can nearly cancel psi(s) and blow up E_loc.
symmetrize = False
# For AF J1 at J2=0 this supplies the Marshall sign: (-1) ** N_up_even.
marshall_sign = False
# Alternative gauge-transformed Hamiltonian sign rule. Do not combine with marshall_sign.
hamiltonian_sign_rule = False
# set both to false since we need to let the vIt LEARN the structure of the ground state without biasing sign rules from gauge theories or known solutions
# In addition these tests were used to see how fast it would have learned the energies if one would have set them equal to these sign rules
if marshall_sign and hamiltonian_sign_rule:
    raise ValueError("Use either marshall_sign or hamiltonian_sign_rule, not both.")

# Sampler / SR. Increase these on the server for a serious run.
seed = 0
n_chains = 1024
burn_in = 120
thin = 20
sample_size_warm = 8
sample_size_main = 64
warm_steps = 5
max_steps = 300
print_every = 1

lr = 8e-3
diag_shift = 1e-2
cg_maxiter = 40
cg_tol = 1e-5
sr_batch_size = 8192# use more e.g 8192 if RAM is available

exact_energy = exact_ground_state_energy_sz0(L, J1=J1, J2=J2)
print(f"Exact benchmark: E0={exact_energy:.8f}, E0/L={exact_energy / L:.8f}")

output_dir = Path("results/netket_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

config = dict(
    L=L,
    J1=J1,
    J2_ratio=J2_ratio,
    J2=J2,
    b=b,
    d=d,
    h=h,
    n_layers=n_layers,
    d_ff=d_ff,
    rbm_hidden=rbm_hidden,
    amp_init_std=amp_init_std,
    phase_init_std=phase_init_std,
    symmetrize=symmetrize,
    marshall_sign=marshall_sign,
    hamiltonian_sign_rule=hamiltonian_sign_rule,
    n_chains=n_chains,
    burn_in=burn_in,
    thin=thin,
    max_steps=max_steps,
    lr=lr,
    diag_shift=diag_shift,
    sr_batch_size=sr_batch_size,
)
print(json.dumps(config, indent=2))

## Build model, sampler, optimizer, Hamiltonian

In [ ]:
key = jax.random.PRNGKey(seed)

model = ViTWaveFunction(
    L=L,
    b=b,
    n_layers=n_layers,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    rbm_hidden=rbm_hidden,
    amp_init_std=amp_init_std,
    phase_init_std=phase_init_std,
    symmetrize=symmetrize,
    marshall_sign=marshall_sign,
    dtype=jnp.float64,
)
params = model.init(key)

sampler = VMCSampler(
    model=model,
    L=L,
    n_chains=n_chains,
    seed=seed,
    print_every=0,
    verbose=False,
)
sampler.reset(params)

sr = SROptimizer(
    model=model,
    lr=lr,
    diag_shift=diag_shift,
    cg_maxiter=cg_maxiter,
    cg_tol=cg_tol,
    batch_size=sr_batch_size,
    seed=seed,
)

H0 = J1J2Hamiltonian(
    L=L,
    J1=J1,
    J2=J2,
    sign_rule=hamiltonian_sign_rule,
    monitor_energy_imag=False,
)

print("real DOF:", count_real_dof(params))
print("parameter norm:", tree_l2_norm(params))
print("RBM kernel dtype:", params["rbm"]["kernel"].dtype)
print("initial sampler state:", sampler.current_state.shape)

## Quick smoke test

In [ ]:
states, _, _ = sampler.sample(params, n_samples=2, burn_in=10, thin=2)
configs = flatten_samples(states)
E_loc = H0.local_energy(model, params, configs).reshape(-1)

if not bool(jnp.all(jnp.isfinite(E_loc))):
    raise FloatingPointError(
        "Non-finite local energies in the smoke test. With a complex ansatz this is usually caused by symmetrize=True."
    )

E_mean = float(jnp.mean(E_loc.real))
E_std = float(jnp.std(E_loc.real))

print("configs:", configs.shape)
print("E_loc:", E_loc.shape, E_loc.dtype)
print("Re(E):", E_mean, "+/-", E_std)
print("Re(E)/L:", E_mean / L, "+/-", E_std / L)
print("Im(E):", float(jnp.mean(E_loc.imag)), "+/-", float(jnp.std(E_loc.imag)))
print("energy stats:", H0.get_last_energy_stats())

## Train

In [ ]:
energy_history = []
energy_per_site_history = []
energy_std_history = []
energy_imag_mean_history = []
energy_imag_std_history = []
delta_norm_history = []
param_norm_history = []

for it in trange(max_steps, desc=f"J2/J1={J2_ratio} SR steps"):
    sample_size = sample_size_warm if it < warm_steps else sample_size_main
    states, _, _ = sampler.sample(params, n_samples=sample_size, burn_in=100, thin=thin)
    configs = flatten_samples(states)

    E_loc = H0.local_energy(model, params, configs).reshape(-1)
    if not bool(jnp.all(jnp.isfinite(E_loc))):
        raise FloatingPointError(
            "Encountered non-finite local energies. The first thing to try is symmetrize=False for this complex ansatz."
        )

    E_mean = float(jnp.mean(E_loc.real))
    E_std = float(jnp.std(E_loc.real))
    E_per_site = E_mean / L
    E_imag_mean = float(jnp.mean(E_loc.imag))
    E_imag_std = float(jnp.std(E_loc.imag))

    params, delta_norm = sr.step(params, configs, E_loc)
    sampler.refresh(params)

    energy_history.append(E_mean)
    energy_per_site_history.append(E_per_site)
    energy_std_history.append(E_std)
    energy_imag_mean_history.append(E_imag_mean)
    energy_imag_std_history.append(E_imag_std)
    delta_norm_history.append(float(delta_norm))
    param_norm_history.append(tree_l2_norm(params))

    if it % print_every == 0:
        print(
            f"step {it:04d} | "
            f"E={E_mean:.8f} | "
            f"E/L={E_per_site:.8f} | "
            f"ImE={E_imag_mean:.3e} +/- {E_imag_std:.3e} | "
            f"delta={float(delta_norm):.3e}"
        )

## Save results

In [ ]:
result_path = output_dir / f"jax_vit_j1j2_L{L}_J2ratio_{J2_ratio}_d_{d}_h_{h}_nl_{n_layers}.npz"
np.savez(
    result_path,
    energy=np.asarray(energy_history),
    energy_per_site=np.asarray(energy_per_site_history),
    energy_std=np.asarray(energy_std_history),
    energy_imag_mean=np.asarray(energy_imag_mean_history),
    energy_imag_std=np.asarray(energy_imag_std_history),
    delta_norm=np.asarray(delta_norm_history),
    param_norm=np.asarray(param_norm_history),
    exact_energy=exact_energy,
    config=json.dumps(config),
)
print("saved:", result_path)


## Plot diagnostics

In [ ]:
if len(energy_history) == 0:
    print("No training history yet.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

    axes[0].plot(energy_per_site_history, marker="o", markersize=3, linewidth=1, label="VMC")
    axes[0].axhline(exact_energy / L, color="k", linestyle="--", linewidth=1, label="exact")
    axes[0].set_xlabel("SR step")
    axes[0].set_ylabel("Mean local energy per site")
    axes[0].set_title("Energy per site")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(energy_imag_mean_history, label="ImE mean")
    axes[1].plot(energy_imag_std_history, label="ImE std")
    axes[1].set_xlabel("SR step")
    axes[1].set_title("Imag(E_loc)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(delta_norm_history, label="SR delta norm")
    axes[2].plot(param_norm_history, label="param norm")
    axes[2].set_xlabel("SR step")
    axes[2].set_title("Update diagnostics")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    fig.tight_layout()
    plt.show()

## Relative error 

In [ ]:
# Create a function to calculate the relative error of the energy compared to the exact energy to plot d/h 

def relative_energy_error(energy_history, exact_energy):
    energy_history = np.asarray(energy_history)
    return np.abs(energy_history - exact_energy) / np.abs(exact_energy)

# Now we can use this function to plot the relative energy error over the SR steps.
relative_error_history = relative_energy_error(energy_history, exact_energy)
plt.figure(figsize=(6, 4))
plt.plot(relative_error_history, marker="o", markersize=3, linewidth=1)
plt.xlabel("SR step")
plt.ylabel("Relative energy error")
plt.title("Relative Energy Error vs SR Step")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

## Final observables

In [ ]:
obs_samples = 20
obs_burn_in = 100
obs_thin = thin

states, _, _ = sampler.sample(params, n_samples=obs_samples, burn_in=obs_burn_in, thin=obs_thin)
configs = np.asarray(flatten_samples(states))

C = spin_correlation(configs)
S = structure_factor_from_C(C)
D = dimer_correlation(configs)
r = np.arange(L)
k = 2 * np.pi * np.arange(L) / L

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(r, C, marker="o", markersize=3)
axes[0].set_xlabel("r")
axes[0].set_title("Spin correlation C(r)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(k, S, marker="o", markersize=3)
axes[1].set_xlabel("k")
axes[1].set_title("Structure factor S(k)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(r, D, marker="o", markersize=3)
axes[2].set_xlabel("r")
axes[2].set_title("Dimer correlation D(r)")
axes[2].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
obs_path = output_dir / f"jax_vit_observables_L{L}_J2ratio_{J2_ratio}_d_{d}_h_{h}_nl_{n_layers}.npz"
np.savez(obs_path, C=C, S=S, D=D, r=r, k=k)
print("saved:", obs_path)

## Optional native NetKet objects

In [ ]:
try:
    graph = H0.netket_graph()
    hilbert = H0.netket_hilbert()
    operator = H0.to_netket_operator()
    nk_sampler = VMCSampler.to_netket_sampler(hilbert, graph, n_chains=4096, d_max=2)
    print(graph)
    print(hilbert)
    print(operator)
    print(nk_sampler)
except ImportError as exc:
    print("NetKet is not installed in this kernel:", exc)